# Set up in cloud

In [1]:
!gsutil -m cp -r gs://seer-models/models/issue_grouping_v1 .

Copying gs://seer-models/models/issue_grouping_v1/.DS_Store...
Copying gs://seer-models/models/issue_grouping_v1/data.pkl...
Copying gs://seer-models/models/issue_grouping_v1/embeddings/README.md...
Copying gs://seer-models/models/issue_grouping_v1/embeddings/config.json...
Copying gs://seer-models/models/issue_grouping_v1/embeddings/1_Pooling/config.json...
Copying gs://seer-models/models/issue_grouping_v1/embeddings/config_sentence_transformers.json...
Copying gs://seer-models/models/issue_grouping_v1/embeddings/modules.json...
Copying gs://seer-models/models/issue_grouping_v1/embeddings/sentence_bert_config.json...
Copying gs://seer-models/models/issue_grouping_v1/embeddings/vocab.json...
Copying gs://seer-models/models/issue_grouping_v1/embeddings/configuration_bert.py...
Copying gs://seer-models/models/issue_grouping_v1/embeddings/special_tokens_map.json...
Copying gs://seer-models/models/issue_grouping_v1/embeddings/merges.txt...
Copying gs://seer-models/models/issue_grouping_v1/

In [4]:
!gsutil -m -o GSUtil:check_hashes=never cp -r gs://seer-models/models/issue_grouping_v1/embeddings/model.safetensors issue_grouping_v1/embeddings/model.safetensors

Copying gs://seer-models/models/issue_grouping_v1/embeddings/model.safetensors...
\ [1/1 files][613.7 MiB/613.7 MiB] 100% Done                                    
Operation completed over 1 objects/613.7 MiB.                                    


In [6]:
!gsutil -m -o GSUtil:check_hashes=never cp -r gs://grouping-data/final_csvs .

Copying gs://grouping-data/final_csvs/synthetic-semi-easy-negatives.csv...
Copying gs://grouping-data/final_csvs/test.csv...
Copying gs://grouping-data/final_csvs/train.csv...
Copying gs://grouping-data/final_csvs/val.csv...


In [1]:
!pip uninstall tensorflow

Add a Colab secret for your `GITHUB_TOKEN`

In [ ]:
import os

os.environ["GITHUB_TOKEN"] = ""
os.environ["WANDB_API_KEY"] = ""

In [3]:
!pip install git+https://${GITHUB_TOKEN}@github.com/getsentry/grouping-trainer.git

  Cloning https://****@github.com/getsentry/grouping-trainer.git to /tmp/pip-req-build-_de3wc_m
  Running command git clone --filter=blob:none --quiet 'https://****@github.com/getsentry/grouping-trainer.git' /tmp/pip-req-build-_de3wc_m
  Resolved https://****@github.com/getsentry/grouping-trainer.git to commit 1914c374126a40bd66b17713ce06f2f4d05d9060
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


# Set up

In [1]:
# !rm -rf output/

In [ ]:
import math

from datetime import datetime
import polars as pl
from sentence_transformers import SentenceTransformerTrainingArguments
from sentence_transformers.training_args import MultiDatasetBatchSamplers
import torch

import grouping_trainer as gt
import utils

In [2]:
timestamp = datetime.now().strftime("%Y-%m-%d-%H-%M-%S")

In [3]:
# assert torch.cuda.is_available()

Some vars to care about

In [ ]:
SAMPLE_TRAIN: int | None = None if torch.cuda.is_available() else 30
SAMPLE_VAL: int | None = None if torch.cuda.is_available() else 20

OUTPUT_DIR = f"./{timestamp}-output"
PER_DEVICE_TRAIN_BATCH_SIZE = 1  # TODO: higher. Needed for L4
GRADIENT_ACCUMULATION_STEPS = 32  # Ideally lower. Needed for L4
PER_DEVICE_EVAL_BATCH_SIZE = 1  # TODO: higher
EVAL_STEPS = 10  # TODO: higher

# Load model

In [5]:
gt.utils._cuda_empty_cache()

In [ ]:
# model_path = "issue_grouping_v1/embeddings"
model_path = "/Users/kdubey/projects/seer/models/issue_grouping_v1/embeddings"
model = gt.utils.SentenceTransformer(str(model_path), trust_remote_code=True)
model.device

device(type='mps', index=0)

In [16]:
assert "layernorm" in repr(model[0].auto_model).lower()
assert "batch" not in repr(model[0].auto_model).lower()

Don't have batch norm. That could mess up stuff for the deduplication strategy.

In [17]:
_ = model.encode("test")

# Load data

We'll make a `dataset_val` for val loss.

In [ ]:
dataset_val = gt.train.df_to_dataset(utils.load_val_df(sample_size=SAMPLE_VAL))
len(dataset_val)

In [ ]:
dataset_dict_train, frac_positive = utils.load_train_dataset_dict(
    sample_size=SAMPLE_TRAIN, min_dataset_size=PER_DEVICE_TRAIN_BATCH_SIZE
)
len(dataset_dict_train)

  0%|          | 0/27 [00:00<?, ?it/s]

# Set up `Trainer`

In [ ]:
evaluator = gt.evaluator.MinPrecisionEvaluator(
    sentences1=list(dataset_val["query_stacktrace_string"]),
    sentences2=list(dataset_val["candidate_stacktrace_string"]),
    labels=[int(record["label"]) for record in dataset_val],
    name="val",
    show_progress_bar=True,
    batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
    truncate_dims=(64, 768),
)

In [14]:
def init_bias(frac_positive: float):
    return math.log(frac_positive / (1 - frac_positive))

In [ ]:
trainer = gt.train.Trainer(
    model=model,
    args=SentenceTransformerTrainingArguments(
        # These should prolly be unchanged
        output_dir=OUTPUT_DIR,
        bf16=torch.cuda.is_bf16_supported(),
        fp16=False,
        dataloader_pin_memory=torch.cuda.is_available(),
        num_train_epochs=1,
        # Save memory
        gradient_checkpointing=True,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        #
        # Datalaoder
        multi_dataset_batch_sampler=MultiDatasetBatchSamplers.PROPORTIONAL,
        # Each iter, pick a project randomly, sample from it.
        # Next iter, pick another project randomly, sample from it, etc.
        per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
        seed=42,  # passed to batch sampler
        #
        # Optimizer
        learning_rate=2e-5,
        learning_rate_mapping={
            # These are important to tune. Higher so that training doesn't get stuck. TODO: check
            r"^log_scale$": 2e-4,
            r"^bias$": 2e-4,
        },
        weight_decay=0.01,
        warmup_ratio=0.1,
        #
        # Eval
        per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
        eval_strategy="steps",
        eval_steps=EVAL_STEPS,
        #
        # Logging
        logging_strategy="steps",
        logging_steps=EVAL_STEPS,  # train loss alongside metrics table
        run_name=f"{timestamp}-grouping-trainer",
        report_to="wandb",
        #
        # Checkpointing
        save_strategy="steps",
        save_steps=EVAL_STEPS // 5,
        save_total_limit=2,
    ),
    #
    # Training
    loss=gt.train.SigmoidPairwiseLoss(
        model,
        bias_init=init_bias(frac_positive),
        matryoshka_dims=[768, 512, 256, 128, 64],
        matryoshka_weights=[2, 1, 1, 0.5, 0.25],
        n_dims_per_step=2,
    ),
    data_collator=gt.train.DefaulDataCollator(tokenize_fn=model.tokenize),
    train_dataset=dataset_dict_train,
    shuffle_within_dataset=False,  # more cache hits in each forward
    #
    # Evaluator
    eval_dataset=dataset_val,  # val loss
    evaluator=evaluator,  # val auc, etc.
    #
    # Other
    # callbacks=[...],  # TrainerCallbacks
)

In [18]:
train_output = trainer.train()

You are using an old version of the checkpointing format that is deprecated (We will also silently ignore `gradient_checkpointing_kwargs` in case you passed it).Please update to the new format on your modeling file. To use the new format, you need to completely remove the definition of the method `_set_gradient_checkpointing` in your model.
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:929: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss,Validation Loss,Val Cosine Accuracy,Val Cosine Accuracy Threshold,Val Cosine F1,Val Cosine F1 Threshold,Val Cosine Precision,Val Cosine Recall,Val Cosine Ap,Val Cosine Mcc
10,2.020200,1.992342,0.756000,0.709827,0.833333,0.707701,0.756824,0.927052,0.872344,0.424622
20,0.801600,1.315565,0.768000,0.582224,0.839309,0.429132,0.745283,0.960486,0.872593,0.434590
30,0.707500,1.254460,0.770000,0.442654,0.843962,0.437011,0.762255,0.945289,0.878242,0.462814


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:929: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:929: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/

Batches:   0%|          | 0/122 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:929: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:929: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/

Batches:   0%|          | 0/122 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:929: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:929: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/

Batches:   0%|          | 0/122 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:929: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


In [19]:
train_output

TrainOutput(global_step=32, training_loss=1.136501893401146, metrics={'train_runtime': 551.681, 'train_samples_per_second': 1.813, 'train_steps_per_second': 0.058, 'total_flos': 0.0, 'train_loss': 1.136501893401146, 'epoch': 1.0})

In [20]:
df_logs = pl.DataFrame(trainer.state.log_history)
df_logs
# df_logs.write_csv("training_logs.csv")

loss,grad_norm,learning_rate,epoch,step,eval_loss,eval_val_cosine_accuracy,eval_val_cosine_accuracy_threshold,eval_val_cosine_f1,eval_val_cosine_f1_threshold,eval_val_cosine_precision,eval_val_cosine_recall,eval_val_cosine_ap,eval_val_cosine_mcc,eval_runtime,eval_samples_per_second,eval_steps_per_second,train_runtime,train_samples_per_second,train_steps_per_second,total_flos,train_loss
f64,f64,f64,f64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2.0202,11.295671,0.000016,0.32,10,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
null,null,null,0.32,10,1.992342,0.756,0.709827,0.833333,0.707701,0.756824,0.927052,0.872344,0.424622,91.4946,5.465,0.689,null,null,null,null,null
0.8016,8.31354,0.000009,0.64,20,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
null,null,null,0.64,20,1.315565,0.768,0.582224,0.839309,0.429132,0.745283,0.960486,0.872593,0.43459,90.4373,5.529,0.697,null,null,null,null,null
0.7075,15.062196,0.000002,0.96,30,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
null,null,null,0.96,30,1.25446,0.77,0.442654,0.843962,0.437011,0.762255,0.945289,0.878242,0.462814,90.4408,5.528,0.697,null,null,null,null,null
null,null,null,1.0,32,null,null,null,null,null,null,null,null,null,null,null,null,551.681,1.813,0.058,0.0,1.136502


In case you OOM'd, consider:

In [ ]:
trainer.per_device_token_budget /= 2
trainer.train(resume_from_checkpoint=True)